In [1]:
"""
Translate raw text with a trained model. Batches data on-the-fly.
"""

import ast
import fileinput
import logging
import math
import os
import sys
import time
from argparse import Namespace
from collections import namedtuple

import numpy as np
import torch

import fairseq
from fairseq import checkpoint_utils, distributed_utils, options, tasks, utils
from fairseq.dataclass.configs import FairseqConfig
from fairseq.dataclass.utils import convert_namespace_to_omegaconf
from fairseq.token_generation_constraints import pack_constraints, unpack_constraints
from fairseq_cli.generate import get_symbols_to_strip_from_output

logging.basicConfig(
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    level=os.environ.get("LOGLEVEL", "INFO").upper(),
    stream=sys.stdout,
)
logger = logging.getLogger("fairseq_cli.interactive")


Batch = namedtuple("Batch", "ids src_tokens src_lengths constraints")
Translation = namedtuple("Translation", "src_str hypos pos_scores alignments")


def make_batches(lines, cfg, task, max_positions, encode_fn):
    def encode_fn_target(x):
        return encode_fn(x)

    if cfg.generation.constraints:
        # Strip (tab-delimited) contraints, if present, from input lines,
        # store them in batch_constraints
        batch_constraints = [list() for _ in lines]
        for i, line in enumerate(lines):
            if "\t" in line:
                lines[i], *batch_constraints[i] = line.split("\t")

        # Convert each List[str] to List[Tensor]
        for i, constraint_list in enumerate(batch_constraints):
            batch_constraints[i] = [
                task.target_dictionary.encode_line(
                    encode_fn_target(constraint),
                    append_eos=False,
                    add_if_not_exist=False,
                )
                for constraint in constraint_list
            ]

    if cfg.generation.constraints:
        constraints_tensor = pack_constraints(batch_constraints)
    else:
        constraints_tensor = None

    tokens, lengths = task.get_interactive_tokens_and_lengths(lines, encode_fn)

    itr = task.get_batch_iterator(
        dataset=task.build_dataset_for_inference(
            tokens, lengths, constraints=constraints_tensor
        ),
        max_tokens=cfg.dataset.max_tokens,
        max_sentences=cfg.dataset.batch_size,
        max_positions=max_positions,
        ignore_invalid_inputs=cfg.dataset.skip_invalid_size_inputs_valid_test,
    ).next_epoch_itr(shuffle=False)
    for batch in itr:
        ids = batch["id"]
        src_tokens = batch["net_input"]["src_tokens"]
        src_lengths = batch["net_input"]["src_lengths"]
        constraints = batch.get("constraints", None)

        yield Batch(
            ids=ids,
            src_tokens=src_tokens,
            src_lengths=src_lengths,
            constraints=constraints,
        )


class FairseqRunner:
    def __init__(self, input_args = None):
        parser = fairseq.options.get_generation_parser()
        parser.add_argument(
            "--arch",
            "-a",
            metavar="ARCH",
            default="wav2vec2",
            help="Model architecture. For constructing tasks that rely on "
            "model args (e.g. `AudioPretraining`)",
        )
        args = options.parse_args_and_arch(parser, input_args)

        cfg = convert_namespace_to_omegaconf(args)

        utils.import_user_module(cfg.common)

        if cfg.interactive.buffer_size < 1:
            cfg.interactive.buffer_size = 1
        if cfg.dataset.max_tokens is None and cfg.dataset.batch_size is None:
            cfg.dataset.batch_size = 1

        assert (
            not cfg.generation.sampling or cfg.generation.nbest == cfg.generation.beam
        ), "--sampling requires --nbest to be equal to --beam"
        assert (
            not cfg.dataset.batch_size
            or cfg.dataset.batch_size <= cfg.interactive.buffer_size
        ), "--batch-size cannot be larger than --buffer-size"

        logger.info(cfg)

        # Fix seed for stochastic decoding
        if cfg.common.seed is not None and not cfg.generation.no_seed_provided:
            np.random.seed(cfg.common.seed)
            utils.set_torch_seed(cfg.common.seed)

        use_cuda = torch.cuda.is_available() and not cfg.common.cpu

        # Setup task, e.g., translation
        task = tasks.setup_task(cfg.task)

        # Load ensemble
        overrides = ast.literal_eval(cfg.common_eval.model_overrides)
        logger.info("loading model(s) from {}".format(cfg.common_eval.path))
        models, _model_args = checkpoint_utils.load_model_ensemble(
            utils.split_paths(cfg.common_eval.path),
            arg_overrides=overrides,
            task=task,
            suffix=cfg.checkpoint.checkpoint_suffix,
            strict=(cfg.checkpoint.checkpoint_shard_count == 1),
            num_shards=cfg.checkpoint.checkpoint_shard_count,
        )

        # Set dictionaries
        src_dict = task.source_dictionary
        tgt_dict = task.target_dictionary

        # Optimize ensemble for generation
        for model in models:
            if model is None:
                continue
            if cfg.common.fp16:
                model.half()
            if use_cuda and not cfg.distributed_training.pipeline_model_parallel:
                model.cuda()
            model.prepare_for_inference_(cfg)

        # Initialize generator
        generator = task.build_generator(models, cfg.generation)

        # Handle tokenization and BPE
        tokenizer = task.build_tokenizer(cfg.tokenizer)
        bpe = task.build_bpe(cfg.bpe)


        # Load alignment dictionary for unknown word replacement
        # (None if no unknown word replacement, empty if no path to align dictionary)
        align_dict = utils.load_align_dict(cfg.generation.replace_unk)

        max_positions = utils.resolve_max_positions(
            task.max_positions(), *[model.max_positions() for model in models]
        )

        self.context = {
            'bpe': bpe,
            'tokenizer': tokenizer,
            'cfg': cfg,
            'task': task,
            'max_positions': max_positions,
            'use_cuda': use_cuda,
            'generator': generator,
            'models': models,
            'src_dict': src_dict,
            'tgt_dict': tgt_dict,
            'align_dict': align_dict,
        }

    def inference(self, input):
        context = self.context

        bpe = context['bpe']
        tokenizer = context['tokenizer']
        cfg = context['cfg']
        task = context['task']
        max_positions = context['max_positions']
        use_cuda = context['use_cuda']
        generator = context['generator']
        models = context['models']
        src_dict = context['src_dict']
        tgt_dict = context['tgt_dict']
        align_dict = context['align_dict']

        def encode_fn(x):
            if tokenizer is not None:
                x = tokenizer.encode(x)
            if bpe is not None:
                x = bpe.encode(x)
            return x

        def decode_fn(x):
            if bpe is not None:
                x = bpe.decode(x)
            if tokenizer is not None:
                x = tokenizer.decode(x)
            return x


        if cfg.generation.constraints:
            logger.warning(
                "NOTE: Constrained decoding currently assumes a shared subword vocabulary."
            )

        if cfg.interactive.buffer_size > 1:
            logger.info("Sentence buffer size: %s", cfg.interactive.buffer_size)
        logger.info("NOTE: hypothesis and token scores are output in base 2")
        logger.info("Type the input sentence and press return:")
        start_id = 0

        # for inputs in buffered_read(input, cfg.interactive.buffer_size):
        # print(input)
        for inputs in [input,]:
            # print(inputs)
            # break
            results = []
            for batch in make_batches([inputs,], cfg, task, max_positions, encode_fn):
                # print(batch)
                bsz = batch.src_tokens.size(0)
                src_tokens = batch.src_tokens
                src_lengths = batch.src_lengths
                constraints = batch.constraints
                if use_cuda:
                    src_tokens = src_tokens.cuda()
                    src_lengths = src_lengths.cuda()
                    if constraints is not None:
                        constraints = constraints.cuda()

                sample = {
                    "net_input": {
                        "src_tokens": src_tokens,
                        "src_lengths": src_lengths,
                    },
                }
                translate_start_time = time.time()
                prefix_tokens = None
                # if cfg.generation.prefix_size > 0:
                #     prefix_tokens = sample["target"][:, : cfg.generation.prefix_size]
                translations = task.inference_step(
                    generator, models, sample, constraints=constraints, prefix_tokens=prefix_tokens, threshold=0.0, soft_attention=False, preRead=2
                )
                translate_time = time.time() - translate_start_time

                list_constraints = [[] for _ in range(bsz)]
                if cfg.generation.constraints:
                    list_constraints = [unpack_constraints(c) for c in constraints]
                for i, (id, hypos) in enumerate(zip(batch.ids.tolist(), translations)):
                    src_tokens_i = utils.strip_pad(src_tokens[i], tgt_dict.pad())
                    constraints = list_constraints[i]
                    results.append(
                        (
                            start_id + id,
                            src_tokens_i,
                            hypos,
                            {
                                "constraints": constraints,
                                "time": translate_time / len(translations),
                            },
                        )
                    )

            outputs = []
            # sort output to match input order
            for id_, src_tokens, hypos, info in sorted(results, key=lambda x: x[0]):
                output_dict = {}
                src_str = ""
                if src_dict is not None:
                    src_str = src_dict.string(src_tokens, cfg.common_eval.post_process)
                    print("S-{}\t{}".format(id_, src_str))
                    output_dict[f'S-{id_}'] = src_str
                    print("W-{}\t{:.3f}\tseconds".format(id_, info["time"]))
                    output_dict[f'W-{id_}'] = info["time"]
                    for constraint in info["constraints"]:
                        print(
                            "C-{}\t{}".format(
                                id_,
                                tgt_dict.string(constraint, cfg.common_eval.post_process),
                            )
                        )

                # Process top predictions
                for hypo in hypos[: min(len(hypos), cfg.generation.nbest)]:
                    hypo_tokens, hypo_str, alignment = utils.post_process_prediction(
                        hypo_tokens=hypo["tokens"].int().cpu(),
                        src_str=src_str,
                        alignment=hypo["alignment"],
                        align_dict=align_dict,
                        tgt_dict=tgt_dict,
                        remove_bpe=cfg.common_eval.post_process,
                        extra_symbols_to_ignore=get_symbols_to_strip_from_output(generator),
                    )
                    detok_hypo_str = decode_fn(hypo_str)
                    score = hypo["score"] / math.log(2)  # convert to base 2
                    # original hypothesis (after tokenization and BPE)
                    print("H-{}\t{}\t{}".format(id_, score, hypo_str))
                    output_dict[f'H-{id_}'] = hypo_str
                    # detokenized hypothesis
                    print("D-{}\t{}\t{}".format(id_, score, detok_hypo_str))
                    output_dict[f'D-{id_}'] = detok_hypo_str
                    print(
                        "P-{}\t{}".format(
                            id_,
                            " ".join(
                                map(
                                    lambda x: "{:.4f}".format(x),
                                    # convert from base e to base 2
                                    hypo["positional_scores"].div_(math.log(2)).tolist(),
                                )
                            ),
                        )
                    )
                    output_dict[f'P-{id_}'] = " ".join(
                                                    map(
                                                        lambda x: "{:.4f}".format(x),
                                                        # convert from base e to base 2
                                                        hypo["positional_scores"].div_(math.log(2)).tolist(),
                                                    )
                                                )
                    if cfg.generation.print_alignment:
                        alignment_str = " ".join(
                            ["{}-{}".format(src, tgt) for src, tgt in alignment]
                        )
                        print("A-{}\t{}".format(id_, alignment_str))

            outputs.append(output_dict)
            # update running id_ counter
            start_id += len(inputs)
        return outputs

2025-01-25 19:17:18 | INFO | fairseq.tasks.text_to_speech | Please install tensorboardX: pip install tensorboardX


In [2]:
input_args=['/mnt/nvme-data1/waris/PSI-TAMU/DST/data/libri_denoising/tokenized/american',
            '--task', 'denoising',
            '--arch', 'transformer_lm',
            '--path', '/mnt/nvme-data1/waris/PSI-TAMU/DST/checkpoints/denoise_pretrain/checkpoint_last.pt',
            '--beam','1',
            '--max-len-a', '1.8', '--max-len-b', '10', '--lenpen', '1', '--min-len', '1',
            '--distributed-world-size', '1',
            ]
runner = FairseqRunner(input_args)

2025-01-25 19:17:20 | INFO | fairseq_cli.interactive | {'_name': None, 'common': {'_name': None, 'no_progress_bar': False, 'log_interval': 100, 'log_format': None, 'log_file': None, 'aim_repo': None, 'aim_run_hash': None, 'tensorboard_logdir': None, 'wandb_project': None, 'azureml_logging': False, 'seed': 1, 'cpu': False, 'tpu': False, 'bf16': False, 'memory_efficient_bf16': False, 'fp16': False, 'memory_efficient_fp16': False, 'fp16_no_flatten_grads': False, 'fp16_init_scale': 128, 'fp16_scale_window': None, 'fp16_scale_tolerance': 0.0, 'on_cpu_convert_precision': False, 'min_loss_scale': 0.0001, 'threshold_loss_scale': None, 'amp': False, 'amp_batch_retries': 2, 'amp_init_scale': 128, 'amp_scale_window': None, 'user_dir': None, 'empty_cache_freq': 0, 'all_gather_list_size': 16384, 'model_parallel_size': 1, 'quantization_config_path': None, 'profile': False, 'reset_logging': False, 'suppress_crashes': False, 'use_plasma_view': False, 'plasma_path': '/tmp/plasma'}, 'common_eval': {'_na

In [3]:
src_tokens = "479 401 40 40 40 82 82 171 451 451 12 160 74 74 168 395 36 141 141 120 93 55 33 67 107 58 106 484 484 173 113 113 171 7 7 297 16 44 451 189 189 198 88 215 215 194 208 36 36 187 227 71 80 80 111 450 450 41 150 63 50 50 50 111 111 172 253 34 170 55 39 160 49 49 100 100 194 84 67 67 101 101 86 86 39 59 323 330 455 455 297 210 210 182 182 182 147 147"
gen_tokens = runner.inference(src_tokens)[0]['H-0']

2025-01-25 19:17:27 | INFO | fairseq_cli.interactive | NOTE: hypothesis and token scores are output in base 2
2025-01-25 19:17:27 | INFO | fairseq_cli.interactive | Type the input sentence and press return:


TypeError: TransformerDecoderLayerBase.forward() got an unexpected keyword argument 'src_seqlen'

In [1]:
args=['/mnt/nvme-data1/waris/PSI-TAMU/DST/data/libri_denoising/tokenized/american',
        '--task', 'denoising',
        '--arch', 'transformer_lm',
        '--path', '/mnt/nvme-data1/waris/PSI-TAMU/DST/checkpoints/denoise_pretrain/checkpoint_last.pt',
        '--beam','1',
        '--max-len-a', '1.8', '--max-len-b', '10', '--lenpen', '1', '--min-len', '1',
        '--distributed-world-size', '1',
        '--is-pretrain'
    ]

In [4]:
import logging
import math
from itertools import chain

import numpy as np
import torch
from fairseq import checkpoint_utils, tasks, utils, options
from fairseq.dataclass.utils import convert_namespace_to_omegaconf
from fairseq.logging.meters import StopwatchMeter


def load_model_and_task(args):
    """
    Load model and task based on the given configuration.
    """
    # if isinstance(cfg, dict):  # Ensure cfg is a DictConfig
    #     cfg = convert_namespace_to_omegaconf(Namespace(**cfg))
    parser = options.get_training_parser()
    args = options.parse_args_and_arch(parser, modify_parser=None, args)
    cfg = convert_namespace_to_omegaconf(args)


    assert cfg.common_eval.path is not None, "--path required for generation!"

    # Setup task
    task = tasks.setup_task(cfg.task)

    # Load models
    print("Loading models...")
    models, saved_cfg = checkpoint_utils.load_model_ensemble(
        utils.split_paths(cfg.common_eval.path),
        task=task,
        strict=(cfg.checkpoint.checkpoint_shard_count == 1),
    )

    # Optimize models for generation
    for model in models:
        if cfg.common.fp16:
            model.half()
        if torch.cuda.is_available() and not cfg.common.cpu:
            model.cuda()
        model.prepare_for_inference_(cfg)

    return task, models, cfg

def generate_sequence(cfg, task, models, input_sequence):
    """
    Generate a translation or prediction for a single sequence.
    """
    # Tokenize input
    tokenizer = task.build_tokenizer(cfg.tokenizer)
    bpe = task.build_bpe(cfg.bpe)
    if tokenizer:
        input_sequence = tokenizer.encode(input_sequence)
    if bpe:
        input_sequence = bpe.encode(input_sequence)

    # Create batch
    src_dict = task.source_dictionary
    src_tokens = src_dict.encode_line(input_sequence, add_if_not_exist=False).unsqueeze(0)
    sample = {
        "net_input": {
            "src_tokens": src_tokens,
            "src_lengths": torch.LongTensor([src_tokens.size(1)]),
        }
    }

    # Generate
    generator = task.build_generator(models, cfg.generation)
    gen_timer = StopwatchMeter()
    gen_timer.start()
    hypos = task.inference_step(generator, models, sample)
    gen_timer.stop(sum(len(hypo[0]["tokens"]) for hypo in hypos))

    # Decode and print results
    tgt_dict = task.target_dictionary
    predictions = []
    for hypo in hypos[0]:
        hypo_tokens = hypo["tokens"].int().cpu()
        hypo_str = tgt_dict.string(hypo_tokens, cfg.common_eval.post_process)
        if bpe:
            hypo_str = bpe.decode(hypo_str)
        if tokenizer:
            hypo_str = tokenizer.decode(hypo_str)
        predictions.append((hypo["score"], hypo_str))

    return predictions, gen_timer.sum

# Example usage in notebook:
# 1. Define configuration dictionary `cfg`.
# 2. Call `load_model_and_task(cfg)` to load the model and task.
# 3. Use `generate_sequence(cfg, task, models, input_sequence)` for interactive generation.


SyntaxError: positional argument follows keyword argument (258688757.py, line 19)

In [3]:
task, models, cfg = load_model_and_task(args)

RuntimeError: 